In [2]:
import pandas as pd
import numpy as np
from gensim.models import Word2Vec
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [3]:
%matplotlib inline

In [4]:
df_movies = pd.read_csv('movies.csv')
df_ratings = pd.read_csv('ratings.csv')

In [5]:
df = pd.merge(df_movies, df_ratings)

In [6]:
df

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,2,5.0,859046895
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0,1303501039
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,8,5.0,858610933
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,11,4.0,850815810
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,14,4.0,851766286
...,...,...,...,...,...,...
105334,148238,A Very Murray Christmas (2015),Comedy,475,3.0,1451213043
105335,148626,The Big Short (2015),Drama,458,4.0,1452014749
105336,148626,The Big Short (2015),Drama,576,4.5,1451687664
105337,148626,The Big Short (2015),Drama,668,4.5,1451148148


In [7]:
df.dropna(inplace=True)

In [8]:
df['movieId'] = df['movieId'].astype(str)

In [9]:
users = df["userId"].unique().tolist()
random.shuffle(users)
users_train = users[:int(0.9 * len(users))]

train_df = df[df['userId'].isin(users_train)]
validation_df = df[~df['userId'].isin(users_train)]

In [10]:
watch_train = []
for user in tqdm(users_train):
    temp = train_df[train_df["userId"] == user]["movieId"].tolist()
    watch_train.append(temp)

100%|██████████| 601/601 [00:00<00:00, 2539.44it/s]


In [11]:
model = Word2Vec(
    window=10,
    sg=1,
    hs=0,
    negative=10,
    alpha=0.03,
    min_alpha=0.0007,
    seed=14
)

model.build_vocab(watch_train, progress_per=200)
model.train(watch_train, total_examples=model.corpus_count, epochs=10)

(846902, 961820)

In [12]:
watch = train_df[["movieId", "title"]]
watch.drop_duplicates(subset="movieId", keep="last", inplace=True)

In [13]:
watch_dict = watch.groupby('movieId')['title'].apply(list).to_dict()

In [14]:
title_to_id = {v[0].lower(): k for k, v in watch_dict.items()}

In [15]:
import re

title_no_year_to_id = {}

pattern = re.compile(r'\s*\(\d{4}\)$')

for full_title, movie_id in title_to_id.items():
    title_no_year = pattern.sub('', full_title).strip()
    title_no_year_to_id[title_no_year] = movie_id

In [16]:
def similar_watch(vector, n=6):
    # Find similar movies
    ms = model.wv.similar_by_vector(vector, topn=n+1)[1:]
    new_ms = []

    for movie in ms:
        title = watch_dict.get(movie[0], ["Unknown Title"])[0]
        new_ms.append((title, movie[1]))
    
    return new_ms

In [17]:
def recommend_by_name_no_year(movie_name, topn=6):
    movie_name = movie_name.lower().strip()
    
    # Exact match in titles without year
    if movie_name not in title_no_year_to_id:
        return f"Movie '{movie_name}' not found (search without year)."
    
    movie_id = title_no_year_to_id[movie_name]
    
    try:
        vector = model.wv[movie_id]
        recommendations = similar_watch(vector, n=topn)
        df_recommend = pd.DataFrame(recommendations, columns=["Recommended Movie", "Similarity Score"])
        return df_recommend
    
    except KeyError:
        return f"Movie ID '{movie_id}' does not have a trained vector."

In [22]:
recommend_by_name_no_year("Grumpier Old Men")

,Recommended Movie,Similarity Score
0,Father of the Bride Part II (1995),0.972413
1,Sabrina (1995),0.971643
2,Waiting to Exhale (1995),0.957136
3,Nixon (1995),0.956494
4,Sudden Death (1995),0.954722
5,Dracula: Dead and Loving It (1995),0.950007
